# Keal Voice — Single-Speaker Fine-Tuning (Google Colab)

This notebook follows the **official Qwen3-TTS single-speaker fine-tuning workflow**. It trains a dedicated speaker identity from your own recordings; it is not just zero-shot cloning.

## Before you start
- Target dataset: **20–30 minutes** of clean speech; 10 minutes is only a small experiment. These are practical targets, not official minimums.
- Split recordings into individual utterances, ideally around 5–15 seconds each. Keep each clip's transcript exact.
- Use one clean 5–15 second reference clip, and keep it identical for every training row.
- Use your own voice or a voice you have explicit permission to train.
- **GPU warning:** the upstream trainer uses BF16 and FlashAttention 2. A100 is the safest option; compatible L4 may work. A T4 is not supported by this upstream training script as-is. Free Colab GPU availability is not guaranteed.

## Google Drive dataset layout
Create `MyDrive/KealVoice/dataset/audio/` and place your utterance files there (`.mp3`, `.wav`, or `.m4a`). The preparation cell automatically creates `metadata.csv` with one row per detected audio file. Fill its `text` column with the exact transcript for each clip, save it, and run the cell again. The notebook uses the first valid dataset clip as the reference if the configured reference filename is not found.


In [ ]:
# Install the upstream Qwen3-TTS training code and dependencies.
!git clone https://github.com/QwenLM/Qwen3-TTS.git /content/Qwen3-TTS
%cd /content/Qwen3-TTS
!pip -q install -e .
!pip -q install accelerate tensorboard soundfile pandas librosa
!apt-get -qq update && apt-get -qq install -y ffmpeg
%cd /content/Qwen3-TTS/finetuning
!nvidia-smi


In [ ]:
from google.colab import drive
from pathlib import Path
import os, json, csv, subprocess, shutil, textwrap
drive.mount('/content/drive')

BASE = Path('/content/drive/MyDrive/KealVoice')
DATASET = BASE / 'dataset'
AUDIO_DIR = DATASET / 'audio'
WORK = Path('/content/keal_voice_finetune')
WAV_DIR = WORK / 'wav_24k_mono'
OUTPUT = BASE / 'finetuned_model'
for p in [AUDIO_DIR, WAV_DIR, OUTPUT]: p.mkdir(parents=True, exist_ok=True)

METADATA_CSV = DATASET / 'metadata.csv'
REFERENCE_FILENAME = 'reference.wav'  # Change to a filename in dataset/audio/; use a clean 5–15 sec clip.
MODEL_ID = 'Qwen/Qwen3-TTS-12Hz-1.7B-Base'  # For a smaller experiment, use Qwen/Qwen3-TTS-12Hz-0.6B-Base.
SPEAKER_NAME = 'keal'
EPOCHS = 3
BATCH_SIZE = 1
LEARNING_RATE = '2e-5'

print('Put audio clips in:', AUDIO_DIR)
print('Create CSV:', METADATA_CSV, 'with columns filename,text')
print('Selected reference:', REFERENCE_FILENAME)


## Dataset preparation rules
1. Prefer one speaker, one microphone, one room, and consistent distance from the microphone.
2. Include normal narration, questions, statements, short and long sentences, varied phonemes, and natural pauses. Do not shout or imitate a character.
3. Avoid music, echo, clipping, background voices, aggressive denoising, compression, and reverb.
4. Each CSV row must match its audio file exactly. Do not use auto-generated transcripts without proofreading.
5. Do not put the entire 30-minute recording into one CSV row. Split it into utterances and transcribe each separately.

Example `metadata.csv`:
```csv
filename,text
clip_001.mp3,Hello everyone. Welcome back to the channel.
clip_002.wav,Today we are going to understand how the internet works.
clip_003.m4a,Let us begin with a simple question.
```


In [ ]:
import pandas as pd
import soundfile as sf
import numpy as np

# Create metadata.csv automatically if it does not exist.
# Every audio file gets a row; you must fill in its exact transcript.
if not METADATA_CSV.exists():
    audio_files = sorted([p for p in AUDIO_DIR.iterdir() if p.is_file() and p.suffix.lower() in {'.wav', '.mp3', '.m4a', '.flac', '.ogg'}])
    template = pd.DataFrame({'filename': [p.name for p in audio_files], 'text': [''] * len(audio_files)})
    template.to_csv(METADATA_CSV, index=False, encoding='utf-8-sig')
    print('Created metadata template:', METADATA_CSV)
    print(f'Found {len(audio_files)} audio files in {AUDIO_DIR}.')
    if not audio_files:
        print('Your audio folder is empty. Upload your split audio clips into the audio folder, then run this cell again.')
    else:
        print('Open metadata.csv in Google Drive and fill the text column with the exact words spoken in each matching file.')
    raise RuntimeError('Metadata template created. Fill in the text column, save metadata.csv, then run this cell again.')

df = pd.read_csv(METADATA_CSV).fillna('')
required = {'filename','text'}
if not required.issubset(df.columns):
    raise ValueError('metadata.csv must contain exactly named columns filename,text (additional columns are okay).')
if df.empty:
    raise ValueError('metadata.csv has no rows. Add one row for every audio clip.')
if (df['filename'].astype(str).str.strip() == '').any():
    raise ValueError('Some rows have an empty filename. Fill every filename cell.')
if (df['text'].astype(str).str.strip() == '').any():
    empty_rows = (df.index[df['text'].astype(str).str.strip() == ''] + 2).tolist()
    raise ValueError(f'Missing transcripts in CSV row(s): {empty_rows[:20]}. Fill the text column with the exact spoken words.')
if len(df) < 20:
    print('WARNING: Fewer than 20 utterances. More varied clean samples are recommended.')

reference_source = AUDIO_DIR / REFERENCE_FILENAME
if not reference_source.exists():
    candidates = [AUDIO_DIR / str(name).strip() for name in df['filename'] if (AUDIO_DIR / str(name).strip()).exists()]
    if candidates:
        reference_source = candidates[0]
        print(f'REFERENCE_FILENAME was not found. Using the first available dataset clip as reference: {reference_source.name}')
    else:
        raise FileNotFoundError(f'No reference audio found. Check that your files are inside {AUDIO_DIR} and filenames match metadata.csv.')

def convert_to_wav(src, dst):
    cmd = ['ffmpeg','-nostdin','-y','-i',str(src),'-vn','-ac','1','-ar','24000','-c:a','pcm_s16le',str(dst)]
    p = subprocess.run(cmd, capture_output=True, text=True)
    if p.returncode:
        raise RuntimeError(f'Could not convert {src.name}: {p.stderr[-1200:]}')

ref_wav = WORK / 'keal_reference.wav'
convert_to_wav(reference_source, ref_wav)
raw_jsonl = WORK / 'train_raw.jsonl'
rows, missing = [], []
for i, row in df.iterrows():
    filename, transcript = str(row.filename).strip(), str(row.text).strip()
    source = AUDIO_DIR / filename
    if not source.exists():
        missing.append(filename); continue
    target = WAV_DIR / f'utt_{i+1:04d}.wav'
    convert_to_wav(source, target)
    info = sf.info(target)
    if info.duration < 1.0:
        raise ValueError(f'{filename} is shorter than 1 second.')
    if info.duration > 20:
        print(f'WARNING: {filename} is {info.duration:.1f}s. Split long utterances for cleaner alignment.')
    rows.append({'audio':str(target),'text':transcript,'ref_audio':str(ref_wav)})
if missing:
    raise FileNotFoundError('Missing audio files: ' + ', '.join(missing[:20]))
if not rows:
    raise ValueError('No valid training rows found.')
with raw_jsonl.open('w',encoding='utf-8') as f:
    for item in rows: f.write(json.dumps(item,ensure_ascii=False)+'\n')
durations = [sf.info(x['audio']).duration for x in rows]
print(f'Prepared {len(rows)} clips; total speech {sum(durations)/60:.1f} minutes; median clip {np.median(durations):.1f}s')
print('Reference clip:', sf.info(ref_wav).duration, 'seconds')
print('Raw training manifest:', raw_jsonl)


In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU found. Select Runtime > Change runtime type > GPU.')
gpu = torch.cuda.get_device_properties(0)
print('GPU:', gpu.name, '| VRAM:', round(gpu.total_memory/1024**3,1), 'GB')
print('BF16 supported:', torch.cuda.is_bf16_supported())
if not torch.cuda.is_bf16_supported():
    raise RuntimeError('The official upstream fine-tuning script uses BF16. This GPU is not suitable for the unmodified trainer; use a BF16-capable GPU such as A100 or compatible L4.')

# Use PyTorch SDPA instead of requiring a separately compiled FlashAttention 2 package.
trainer_file = Path('/content/Qwen3-TTS/finetuning/sft_12hz.py')
trainer_code = trainer_file.read_text(encoding='utf-8')
trainer_code = trainer_code.replace('attn_implementation="flash_attention_2"', 'attn_implementation="sdpa"')
trainer_file.write_text(trainer_code, encoding='utf-8')
print('Trainer attention backend set to PyTorch SDPA.')


In [ ]:
RAW_JSONL = str(raw_jsonl)
TRAIN_JSONL = str(WORK / 'train_with_codes.jsonl')
!python prepare_data.py --device cuda:0 --tokenizer_model_path Qwen/Qwen3-TTS-Tokenizer-12Hz --input_jsonl "$RAW_JSONL" --output_jsonl "$TRAIN_JSONL"
print('Tokenized training data:', TRAIN_JSONL)


## Fine-tune
Start with 3 epochs and batch size 1. Watch GPU memory. If it runs out of memory, this upstream full-model trainer may require a larger GPU; do not assume changing the batch size alone will make a T4 work. Checkpoints are saved in Google Drive.

In [ ]:
import subprocess, shlex
cmd = [
    'python','sft_12hz.py',
    '--init_model_path',MODEL_ID,
    '--output_model_path',str(OUTPUT),
    '--train_jsonl',TRAIN_JSONL,
    '--batch_size',str(BATCH_SIZE),
    '--lr',LEARNING_RATE,
    '--num_epochs',str(EPOCHS),
    '--speaker_name',SPEAKER_NAME,
]
print('Running:', ' '.join(shlex.quote(x) for x in cmd))
subprocess.run(cmd, cwd='/content/Qwen3-TTS/finetuning', check=True)
print('Training complete. Checkpoints:', list(OUTPUT.glob('checkpoint-*')))


In [ ]:
import torch, soundfile as sf
from qwen_tts import Qwen3TTSModel
checkpoints = sorted(OUTPUT.glob('checkpoint-*'), key=lambda p:int(p.name.split('-')[-1]))
if not checkpoints:
    raise FileNotFoundError('No checkpoint found. Complete training first.')
BEST_CHECKPOINT = checkpoints[-1]
tts = Qwen3TTSModel.from_pretrained(
    str(BEST_CHECKPOINT), device_map='cuda:0', dtype=torch.bfloat16,
    attn_implementation='flash_attention_2'
)
TEST_TEXT = 'Hello everyone. This is a test of my personal voice model.'
wavs, sr = tts.generate_custom_voice(text=TEST_TEXT, speaker=SPEAKER_NAME, language='English')
test_path = BASE / 'keal_finetuned_test.wav'
sf.write(test_path, wavs[0], sr)
print('Saved test audio:', test_path)
from IPython.display import Audio, display
display(Audio(str(test_path)))
